In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from scipy.stats import rankdata
from sklearn.metrics import roc_auc_score
from scipy.optimize import minimize
from scipy.special import expit as sigmoid
from scipy.special import logit
from pathlib import Path
import sys

COMP_NAME = "g2net-gravitational-wave-detection"

INPUT_PATH = Path(f"/mnt/storage_dimm2/kaggle_data/{COMP_NAME}/")
OUTPUT_PATH = Path(f"/mnt/storage_dimm2/kaggle_output/{COMP_NAME}/")

SAVE_FILES = False

In [ ]:
sys.path.append(f"/home/anjum/kaggle/{COMP_NAME}")

from src.stacking import OptimizeAUC

In [ ]:
gt = pd.read_csv(INPUT_PATH / "training_labels.csv")


# With TTA
oof_files = [
    "oofs_20210830-191530_CV-0.87733.csv", # ResNet200d
    "oofs_20210901-130913_CV-0.87749.csv", # EfficientNet B5
    "oofs_20210902-200442_CV-0.87733.csv", # EfficientNet V2 M
    "oofs_20210915-003751_CV-0.87393.csv", # 1D CNN
    "oofs_20210915-072731_CV-0.87707.csv", # EfficientNet B5 CWT
    "oofs_20210918-112751_CV-0.87870.csv", # EfficientNet B5 PL
    "oofs_20210920-134112_CV-0.87875.csv", # EfficientNet B5 PL (selective harden)
]

oof_paths = [OUTPUT_PATH / f.split("_")[1] / f for f in oof_files]
sub_paths = [OUTPUT_PATH / f.split("_")[1] / f.replace("oofs", "sub") for f in oof_files]

oof_list = [pd.read_csv(p) for p in oof_paths]
sub_list = [pd.read_csv(p) for p in sub_paths]

oof1 = oof_list[0]
sub1 = sub_list[0]

oof_df = pd.DataFrame({f"oof{i+1}": o["prediction"] for i, o in enumerate(oof_list)})
oof_df[["id", "target", "fold"]] = oof1[["id", "target", "fold"]]


sub_df = pd.DataFrame({f"sub{i+1}": s["target"] for i, s in enumerate(sub_list)})
sub_df["id"] = sub1["id"]

In [ ]:
oof_df.describe()

In [ ]:
sub_df.describe()

In [ ]:
oof_list[1].loc[oof_list[1]["prediction"].idxmax()]

In [ ]:
oof_list[1].loc[oof_list[1]["prediction"].idxmin()]

In [ ]:
plt.figure(figsize=(8,6))
for o in oof_df.columns[:-3]:
    plt.hist(oof_df[o], bins=100, alpha=0.5, range=[-5, 5], label=o)
plt.legend(loc='upper right');

In [ ]:
plt.figure(figsize=(8,6))
for s in sub_df.columns[:-1]:
    plt.hist(sub_df[s], bins=100, alpha=0.5, range=[-5, 5], label=s)
plt.legend(loc='upper right');

In [ ]:
oof_df["mean"] = sigmoid(oof_df[oof_df.columns[:-3]].mean(axis=1))
oof_df[oof_df["target"] == 0]["mean"].hist(bins=100, alpha=0.5);
oof_df[oof_df["target"] == 1]["mean"].hist(bins=100, alpha=0.5);

In [ ]:
sigmoid(oof_df[oof_df["target"] == 0]["oof2"]).hist(bins=100, alpha=0.5);
sigmoid(oof_df[oof_df["target"] == 1]["oof2"]).hist(bins=100, alpha=0.5);

In [ ]:
sigmoid(oof_df[oof_df["target"] == 0]["oof6"]).hist(bins=100, alpha=0.5);
sigmoid(oof_df[oof_df["target"] == 1]["oof6"]).hist(bins=100, alpha=0.5);
sigmoid(oof_df[oof_df["target"] == 1]["oof7"]).hist(bins=100, alpha=0.5);

In [ ]:
sigmoid(oof_df[oof_df["target"] == 0]["oof2"]).hist(bins=100, alpha=0.5);
sigmoid(oof_df[oof_df["target"] == 0]["oof6"]).hist(bins=100, alpha=0.5);
sigmoid(oof_df[oof_df["target"] == 0]["oof7"]).hist(bins=100, alpha=0.5);

In [ ]:
sigmoid(oof_df[oof_df["target"] == 1]["oof2"]).hist(bins=100, alpha=0.5);
sigmoid(oof_df[oof_df["target"] == 1]["oof6"]).hist(bins=100, alpha=0.5);

In [ ]:
oof_df[oof_df.columns[:-3]].corr()

# Sum, rank & power ensemble

In [ ]:
def score_oof(oof, full=False):
    if full:
        return roc_auc_score(oof["target"], oof["prediction"])
    else:
        scores = [roc_auc_score(df["target"], df["prediction"]) for _, df in oof.groupby("fold")]
        return np.mean(scores)

In [ ]:
for o in oof_list:
    print(score_oof(o), score_oof(o, True))

In [ ]:
sum_preds = oof1.copy()
sum_preds["prediction"] = 0

for o in oof_list:
    sum_preds["prediction"] += o["prediction"]
    
score_oof(sum_preds)

In [ ]:
prob_sum_preds = oof1.copy()
prob_sum_preds["prediction"] = 0

for o in oof_list:
    prob_sum_preds["prediction"] += sigmoid(o["prediction"])
    
score_oof(prob_sum_preds)

In [ ]:
rank_preds = oof1.copy()
rank_preds["prediction"] = 0

for o in oof_list:
    rank_preds["prediction"] += o["prediction"].rank()
    
score_oof(rank_preds)

In [ ]:
for power in [0.25, 0.5, 1.0, 1.5, 2, 4, 8]:
    pow_preds = oof1.copy()
    pow_preds["prediction"] = 0
    for o in oof_list:
        pow_preds["prediction"] += sigmoid(o["prediction"]) ** power

    print(f"{power:0.1f}, {score_oof(pow_preds):0.6f}")

power = 1.5
pow_preds = oof1.copy()
pow_preds["prediction"] = 0
for o in oof_list:
    pow_preds["prediction"] += sigmoid(o["prediction"]) ** power

In [ ]:
# Make a summed submission
sum_sub = sub1.copy()
sum_sub["target"] = 0
for s in sub_list:
    sum_sub["target"] += s["target"]

if SAVE_FILES:
    sum_sub.to_csv(f"stack_subs/summed_ensemble_{score_oof(sum_preds):0.5f}.csv", index=False)

sum_sub.head()

In [ ]:
# Make a ranked submission
rank_sub = sub1.copy()
rank_sub["target"] = 0
for s in sub_list:
    rank_sub["target"] += s["target"].rank()

if SAVE_FILES:
    rank_sub.to_csv(f"stack_subs/ranked_ensemble_{score_oof(rank_preds):0.5f}.csv", index=False)

rank_sub.head()

In [ ]:
# Make a power submission
pow_sub = sub1.copy()
pow_sub["target"] = 0
for s in sub_list:
    pow_sub["target"] += sigmoid(s["target"]) ** power

if SAVE_FILES:
    pow_sub.to_csv(f"stack_subs/power_ensemble_{score_oof(pow_preds):0.5f}.csv", index=False)

pow_sub.head()

# Use `LogisticRegressionCV`

In [ ]:
from sklearn.linear_model import LogisticRegressionCV

In [ ]:
cols = oof_df.columns[:len(oof_files)]

# opt = LogisticRegressionCV(Cs=50, n_jobs=16, scoring="roc_auc", penalty="elasticnet", solver="saga", l1_ratios=np.linspace(0, 1, 10))
# opt.fit(oof_df[cols], oof_df["target"])
# opt.scores_[1].mean(0).max()

# Use `OptimizeAUC`

In [ ]:
# scores = []
# cols = oof_df.columns[:len(oof_files)]

# # poly = PolynomialFeatures(2)

# for fold in range(5):
#     trn_df = oof_df.query(f"fold != {fold}").copy()
#     val_df = oof_df.query(f"fold == {fold}").copy()
    
#     trn_X = sigmoid(trn_df[cols])
#     val_X = sigmoid(val_df[cols])

#     opt = OptimizeAUC("Nelder-Mead")
#     opt.fit(trn_X, trn_df["target"])
#     y_pred = opt.predict(val_df[cols])
# #     y_pred = opt.predict_proba(val_X)[:, 1]
#     scores.append(roc_auc_score(val_df["target"], y_pred))
#     print(f"Fold {fold}, validation score: {scores[-1]:0.6f}") # , opt.res_.x)
#     print()
    
# print()
# print(f"Mean score: {np.mean(scores):0.6f}")

# # Fit on full OOFs and predict on test
# opt = OptimizeAUC()
# opt.fit(oof_df[cols], oof_df["target"])
# print()
# print("Weights", opt.coef_)
# test_preds = opt.predict(sub_df.iloc[:, :-1])


# Mean score: 0.879274
# Mean score: 0.879286

# Create Pseudo-labels

In [ ]:
test_paths = [OUTPUT_PATH / f.split("_")[1] / f.replace("oofs", "test") for f in oof_files]

test_list = [pd.read_csv(p) for p in test_paths]

test1 = test_list[0]
pl_df = pd.DataFrame(test_list[0][["id"]])

for i in range(5):
    fold_labels = 0
    for t in test_list:
        fold_labels += t[f"fold_{i}"]
        
    pl_df[f"fold_{i}"] = sigmoid(fold_labels / len(test_list))
    

pl_df["mean"] = pl_df[pl_df.columns[1:]].mean(axis=1)

if SAVE_FILES:
    pl_df.to_csv(INPUT_PATH / "pl_5_model_ensemble.csv", index=False)
        
pl_df.head()

In [ ]:
pl_df_filtered = pl_df # .query("(mean < 0.4) | (mean > 0.8 & mean < 0.99)")

plt.figure(figsize=(8,6))
for c in pl_df.columns[1:]:
    plt.hist(pl_df_filtered[c], bins=100, alpha=0.5, label=c)
plt.legend(loc='upper right');

In [ ]:
# Harden PLs
cols = [f"fold_{i}" for i in range(5)]
pl_df_hard = pl_df.copy()
idxs = pl_df_hard["mean"] > 0
pl_df_hard.loc[idxs, cols] = sigmoid(logit(pl_df_hard.loc[idxs, cols]) * 2)


plt.figure(figsize=(8,6))
for c in pl_df.columns[1:]:
    plt.hist(pl_df_hard[c], bins=100, alpha=0.5, label=c)
plt.legend(loc='upper right');

In [ ]:
oof_df["mean_hard"] = sigmoid(logit(oof_df["mean"]) * 2)

print(oof_df["mean_hard"].min(), oof_df["mean_hard"].max())

oof_df[oof_df["target"] == 0]["mean_hard"].hist(bins=100, alpha=0.5);
oof_df[oof_df["target"] == 1]["mean_hard"].hist(bins=100, alpha=0.5);

In [ ]:
oof_df[(oof_df["target"] == 1) & (oof_df["mean"] < 0.5)].shape[0] / oof_df[oof_df["target"] == 1].shape[0]

# Linear weighting

In [ ]:
# def calc_weights(df1, df2):
    
#     def func(weight):
#         return score_oof(weight_dfs(df1, df2, weight))

#     weights = []
#     sweep = np.linspace(0, 1, 101)

#     scores = np.array([func(w) for w in sweep])
#     weights.append(sweep[np.argmax(scores)])
        
#     return weights


# def weight_dfs(df1, df2, w):
#     df_out = df1.copy()
#     df_out["prediction"] = w * df1["prediction"] + (1 - w) * df2["prediction"]
#     return df_out


# def stack(df1, df2):
#     weights = calc_weights(df1, df2)
#     print(list(np.round(weights, 2)))
    
#     sub = weight_dfs(df1, df2, weights)
#     print(score_df(sub, gt))
#     return sub

In [ ]:
# calc_weights(oof1, oof3)